# Input dataset check — clarification (issue #1)

This notebook validates the items required under **issue #1** on the Studio VM.

**Step 1 (this notebook):** list and analyze the input Sentinel product *metadata* and **all data** shipped with the imagery;
answer in particular whether **gain/offset values for radiometric correction are provided with the product**
(issue #1 · C1).

> Run: attach the kernel to JupyterHub/Studio VM. To inspect a specific product, set
> the `S2_INPUT_PRODUCT` environment variable; otherwise the auto-discovery below + `PRODUCT_INDEX`
> is used.

In [ ]:
import os 
import sys
from pathlib import Path

current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")
work_dir = Path(current_dir + "/s2-msi-raw-generator")
os.chdir(work_dir)
print(f"Changed working directory to: {work_dir}")


In [ ]:
# One-time kernel setup — safe to re-run. RESTART kernel after setup.



def _find_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists() and (base / "s2_msi_raw_generator").is_dir():
            return base
    return None

_need_pkg = _need_deps = False
try:
    import s2_msi_raw_generator  # noqa: F401
except Exception:
    _need_pkg = True
try:
    import zarr        # noqa: F401
    import numpy       # noqa: F401
    import matplotlib  # noqa: F401
except Exception:
    _need_deps = True

_repo = _find_repo()
if _need_pkg and _repo is not None:
    !{sys.executable} -m pip install --quiet -e {_repo} --no-deps
elif _need_pkg:
    print("WARNING: repo root (pyproject.toml + s2_msi_raw_generator) not found; is PYTHONPATH set?")
if _need_deps:
    !{sys.executable} -m pip install --quiet zarr numpy matplotlib
    !{sys.executable} -m pip install --quiet --force-reinstall --no-deps zarr matplotlib
if _need_pkg or _need_deps:
    print("installed/fixed — RESTART kernel and re-run all cells")
else:
    print(f"environment OK (repo: {_repo})")

In [12]:
# 22) List ALL S2 MSI calibration ADFs in the bucket (inventory only, no download)
import json
import os
import re
from collections import defaultdict
from pathlib import Path

import boto3
from botocore.config import Config

_cands = [os.environ.get("EOPF_SECRETS"),
                      os.path.expanduser("~/.eopf/secrets.json")]
SECRETS = next((p for p in _cands if p and os.path.exists(p)), None)
assert SECRETS, "no .eopf/secrets.json found (set EOPF_SECRETS)"
e = json.loads(Path(SECRETS).read_text())["s2input"]
s3 = boto3.client("s3", endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
                  region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
                  aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 3}))
BUCKET, PREFIX = "dpr-common", "ADF-S02MSI/"

# parse: S2A_ADF_<TYPE>_<start>_<stop>_<creation>.<ext>
ADF_RE = re.compile(r"(?P<plat>S2[ABCD])_ADF_(?P<type>[A-Z0-9]+)_(?P<start>\d{8})T")

# discover every object under the ADF prefix
objs, tok = [], None
while True:
    kw = dict(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=1000)
    if tok:
        kw["ContinuationToken"] = tok
    r = s3.list_objects_v2(**kw)
    objs += [(o["Key"], o["Size"]) for o in r.get("Contents", []) if not o["Key"].endswith("/")]
    if r.get("IsTruncated"):
        tok = r.get("NextContinuationToken")
    else:
        break

# inventory grouped by ADF type (type -> files with epoch + size)
by_type = defaultdict(list)
for k, sz in objs:
    m = ADF_RE.search(k.split("/")[-1])
    t = m.group("type") if m else "OTHER"
    by_type[t].append((k, sz))

total = sum(sz for _, sz in objs)
print(f"found {len(objs)} objects under {BUCKET}/{PREFIX}  ({total/2**20:.1f} MB total)\n")
for t in sorted(by_type):
    items = sorted(by_type[t], key=lambda x: x[0])
    tsz = sum(sz for _, sz in items) / 2**20
    print(f"  {t:8} x{len(items):>3}  {tsz:8.1f} MB")
    for k, sz in items:
        m = ADF_RE.search(k.split("/")[-1])
        ep = m.group("start") if m else "?"
        print(f"       {sz/2**20:7.1f} MB  {ep}  {k.split('/')[-1]}")


found 3858 objects under dpr-common/ADF-S02MSI/  (7183.2 MB total)

  ATMSA    x  2       0.0 MB
           0.0 MB  20240417  S2A_ADF_ATMSA_20240417T000000_21000101T000000_20240417T000000.json
           0.0 MB  20240417  S2B_ADF_ATMSA_20240417T000000_21000101T000000_20240417T000000.json
  BLIND    x  2       0.5 MB
           0.3 MB  20240417  S2A_ADF_BLIND_20240417T000000_21000101T000000_20240417T000000.json
           0.3 MB  20240417  S2B_ADF_BLIND_20240417T000000_21000101T000000_20240417T000000.json
  DATAT    x  4       0.1 MB
           0.0 MB  20240417  S2A_ADF_DATAT_20240417T000000_21000101T000000_20240417T000000.json
           0.0 MB  20240722  S2A_ADF_DATAT_20240722T000000_21000101T000000_20240722T000000.json
           0.0 MB  20240417  S2B_ADF_DATAT_20240417T000000_21000101T000000_20240417T000000.json
           0.0 MB  20240722  S2B_ADF_DATAT_20240722T000000_21000101T000000_20240722T000000.json
  GPARE    x  2       0.1 MB
           0.0 MB  20240417  S2A_ADF_GPARE_20240

In [14]:
# 23) VM filesystem scan: S2 entries in the window [20240417 , 20240417+40d]
#
# Our ESA R2EQOG ADF (and the RABCA set) is dated 2024-04-17. For a temporally-valid,
# ESA-only correction we want an S2 acquisition WITHIN ~40 days AFTER that date. This cell
# walks /home/jovyan/shared and /home/jovyan/validation-data, keeps any entry whose name
# contains "S2" and whose date falls in [2024-04-17, 2024-05-27], sorted by date.
import os
import re
from datetime import date, timedelta

START = date(2024, 4, 17)           # 20240417
END = START + timedelta(days=40)    # 20240527
ROOTS = ["/home/jovyan/shared", "/home/jovyan/validation-data"]
PRUNE = (".SAFE", ".zarr")          # don't descend into product internals (granules/chunks)


def _sens(name):
    for pat in (r"_S(\d{8})T", r"_V(\d{8})T", r"(\d{8})T\d{6}", r"(20\d{6})"):
        m = re.search(pat, name)
        if m:
            s = m.group(1)
            try:
                return date(int(s[:4]), int(s[4:6]), int(s[6:8]))
            except ValueError:
                pass
    return None


hits = {}   # full path -> date (only entries inside the window)
for root in ROOTS:
    if not os.path.isdir(root):
        print(f"[skip] not found: {root}")
        continue
    n = 0
    for dp, dns, fns in os.walk(root):
        for nm in dns + fns:
            n += 1
            if "S2" not in nm and "S02" not in nm:
                continue
            d = _sens(nm)
            if d and START <= d <= END:
                hits[os.path.join(dp, nm)] = d
        dns[:] = [d for d in dns if not d.endswith(PRUNE)]
    print(f"[scan] {root}: {n} entries")

print(f"\n=== S2 entries in [{START} , {END}] on the VM filesystem (date | +days | path) ===")
rows = sorted(((d, (d - START).days, p) for p, d in hits.items()))
for d, off, p in rows:
    print(f"  {str(d):12} +{off:>2}d  {p}")
if not rows:
    print("  (no S2 entries dated within the 40-day window)")


[scan] /home/jovyan/shared: 539560 entries
[scan] /home/jovyan/validation-data: 823052 entries

=== S2 entries in [2024-04-17 , 2024-05-27] on the VM filesystem (date | +days | path) ===
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_ATMSA_20240417T000000_21000101T000000_20240417T000000.json
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_BLIND_20240417T000000_21000101T000000_20240417T000000.json
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_DATAT_20240417T000000_21000101T000000_20240417T000000.json
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_GPARE_20240417T000000_21000101T000000_20240417T000000.json
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_INTDP_20240417T000000_21000101T000000_20240417T000000.json
  2024-04-17   + 0d  /home/jovyan/validation-data/data-store/esa-adf/S2A_ADF_INVLO_20240417T000000_21000101T000000_20240417T000000.js

In [ ]:
# 24) Bucket scan: S2 keys in the window [20240417 , 20240417+40d]
#
# Same window as cell 23 but against the OVH S3 buckets reachable from the VM: dpr-common
# (via s3input) and dpr-s2-input (via s2input). Keys are collapsed to the deepest path
# segment that still contains "S2" (the product folder); kept only if dated in the window.
import json
import os
import re
from datetime import date, timedelta

import boto3
from botocore.config import Config

_cands = [os.environ.get("EOPF_SECRETS"),
                      os.path.expanduser("~/.eopf/secrets.json")]
SECRETS = next((p for p in _cands if p and os.path.exists(p)), None)
assert SECRETS, "no .eopf/secrets.json found (set EOPF_SECRETS)"
raw = json.loads(open(SECRETS).read())
START = date(2024, 4, 17)           # 20240417
END = START + timedelta(days=40)    # 20240527


def _client(alias):
    e = raw[alias]
    return boto3.client(
        "s3",
        endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
        region_name=e.get("region_name", "sbg"),
        aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
        aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
        config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 2}),
    )


def _sens(name):
    for pat in (r"_S(\d{8})T", r"_V(\d{8})T", r"(\d{8})T\d{6}", r"(20\d{6})"):
        m = re.search(pat, name)
        if m:
            s = m.group(1)
            try:
                return date(int(s[:4]), int(s[4:6]), int(s[6:8]))
            except ValueError:
                pass
    return None


def _root(key):
    # deepest path segment that still contains S2/S02 = the product folder
    parts = key.split("/")
    idx = max((i for i, s in enumerate(parts) if "S2" in s or "S02" in s), default=len(parts) - 1)
    return "/".join(parts[: idx + 1])


def _sat(k):
    if re.search(r"S2A|S02A", k): return "S2A"
    if re.search(r"S2B|S02B", k): return "S2B"
    if re.search(r"S2C|S02C", k): return "S2C"
    return "?"


TARGETS = [
    ("s2input", "dpr-common", ["S2AMSIdataset/", "S2BMSIdataset/"]),
    ("s2input", "dpr-s2-input", ["Validation/"]),
]

hits = {}   # (bucket, root) -> (date, sat)  — only entries inside the window
for alias, bucket, prefixes in TARGETS:
    try:
        s3 = _client(alias)
    except KeyError:
        print(f"[{alias}] not in secrets — skip")
        continue
    for pfx in prefixes:
        try:
            tok, n = None, 0
            while True:
                kw = dict(Bucket=bucket, Prefix=pfx, MaxKeys=1000)
                if tok:
                    kw["ContinuationToken"] = tok
                r = s3.list_objects_v2(**kw)
                for o in r.get("Contents", []):
                    k = o["Key"]; n += 1
                    if "S2" not in k and "S02" not in k:
                        continue
                    d = _sens(k.split("/")[-1]) or _sens(k)
                    if not (d and START <= d <= END):
                        continue
                    root = _root(k)
                    key = (bucket, root)
                    if key not in hits or d < hits[key][0]:
                        hits[key] = (d, _sat(k))
                if r.get("IsTruncated"):
                    tok = r.get("NextContinuationToken")
                else:
                    break
            print(f"[{alias}] {bucket}/{pfx}: scanned {n} keys")
        except Exception as exc:
            code = getattr(exc, "response", {}).get("Error", {}).get("Code", type(exc).__name__)
            print(f"[{alias}] {bucket}/{pfx}: {code}")

print(f"\n=== S2 keys in [{START} , {END}] in the buckets (date | +days | sat | location) ===")
rows = sorted(((d, (d - START).days, sat, f"{b}/{root}") for (b, root), (d, sat) in hits.items()))
for d, off, sat, loc in rows:
    print(f"  {str(d):12} +{off:>2}d  {sat:4} {loc}")
if not rows:
    print("  (no S2 keys dated within the 40-day window)")


In [13]:
# 25) Analyze calibration ADF content for the 20240430 & 20240722 epochs (in-memory, no download)
import json
import os
import re
from collections import defaultdict

import numpy as np
import boto3
from botocore.config import Config

_cands = [os.environ.get("EOPF_SECRETS"),
                      os.path.expanduser("~/.eopf/secrets.json")]
SECRETS = next((p for p in _cands if p and os.path.exists(p)), None)
assert SECRETS, "no .eopf/secrets.json found (set EOPF_SECRETS)"
e = json.loads(Path(SECRETS).read_text())["s2input"]
s3 = boto3.client("s3", endpoint_url=(e.get("endpoint_url") or "https://s3.sbg.io.cloud.ovh.net").rstrip("/"),
                  region_name=e.get("region_name", "sbg"),
                  aws_access_key_id=e.get("access_key") or e.get("access_key_id") or e.get("key"),
                  aws_secret_access_key=e.get("secret") or e.get("secret_access_key"),
                  config=Config(s3={"addressing_style": "path"}, connect_timeout=20, retries={"max_attempts": 3}))
BUCKET, PREFIX = "dpr-common", "ADF-S02MSI/"
EPOCHS = ["20240430", "20240722"]
TYPES = None    # e.g. {"REQOG"} to restrict; None = every ADF type present at those epochs
ADF_RE = re.compile(r"(?P<plat>S2[ABCD])_ADF_(?P<type>[A-Z0-9]+)_(?P<start>\d{8})T")

# 1) find the ADF files at those epochs
objs, tok = [], None
while True:
    kw = dict(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=1000)
    if tok:
        kw["ContinuationToken"] = tok
    r = s3.list_objects_v2(**kw)
    objs += [(o["Key"], o["Size"]) for o in r.get("Contents", []) if not o["Key"].endswith("/")]
    if r.get("IsTruncated"):
        tok = r.get("NextContinuationToken")
    else:
        break

targets = []
for k, sz in objs:
    m = ADF_RE.search(k.split("/")[-1])
    if m and m.group("start") in EPOCHS and (TYPES is None or m.group("type") in TYPES):
        targets.append((k, sz, m.group("plat"), m.group("type"), m.group("start")))
targets.sort(key=lambda x: (x[3], x[2], x[4]))
print(f"{len(targets)} ADF files at {EPOCHS}:")
for k, sz, plat, typ, ep in targets:
    print(f"   {sz/2**20:7.1f} MB  {plat} {typ} {ep}  {k.split('/')[-1]}")


def _read_json(key):
    return json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())


def _summ(node):
    try:
        a = np.asarray(node.get("data"), dtype=float)
    except Exception:
        return None, "(non-numeric)"
    if a.size == 0:
        return a.shape, "empty"
    f = a[np.isfinite(a)]
    return a.shape, (f"min={f.min():.4g} max={f.max():.4g} mean={f.mean():.4g} "
                     f"nan={a.size - f.size}") if f.size else "all-nan"


# 2) per-file structural analysis
store = {}   # (type, plat, epoch) -> parsed dict
for k, sz, plat, typ, ep in targets:
    print(f"\n===== {plat} {typ} {ep}  ({sz/2**20:.1f} MB) =====")
    try:
        d = _read_json(k)
    except Exception as exc:
        print("  read failed:", type(exc).__name__, exc)
        continue
    store[(typ, plat, ep)] = d
    print("  top-level keys:", list(d.keys()))
    at = d.get("attrs", {}) or {}
    if at:
        print("  attrs:", {kk: at[kk] for kk in list(at)[:12]})
    dv = d.get("data_vars", {}) or {}
    print(f"  data_vars: {len(dv)}")
    by_band = defaultdict(list)
    for name in dv:
        by_band[name.split("/")[0] if "/" in name else "(root)"].append(name.split("/")[-1])
    for band in sorted(by_band):
        print(f"    band {band:5}: {sorted(set(by_band[band]))}")
    for name in list(dv)[:6]:
        shp, stat = _summ(dv[name])
        print(f"      {name:22} dims={dv[name].get('dims')} shape={shp}  {stat}")

# 3) epoch-to-epoch change for each (type, platform) present at BOTH epochs
print("\n" + "=" * 60)
print(f"CHANGE across epochs  {EPOCHS[0]} -> {EPOCHS[1]}")
for (typ, plat) in sorted({(t, p) for (t, p, _) in store}):
    d1, d2 = store.get((typ, plat, EPOCHS[0])), store.get((typ, plat, EPOCHS[1]))
    if not (d1 and d2):
        continue
    dv1, dv2 = d1.get("data_vars", {}), d2.get("data_vars", {})
    shared = sorted(set(dv1) & set(dv2))
    diffs = []
    for name in shared:
        try:
            a1 = np.asarray(dv1[name]["data"], float)
            a2 = np.asarray(dv2[name]["data"], float)
        except Exception:
            continue
        if a1.shape != a2.shape:
            print(f"   {plat} {typ} {name}: shape {a1.shape} -> {a2.shape}")
            continue
        msk = np.isfinite(a1) & np.isfinite(a2)
        if not msk.any():
            continue
        absd = np.abs(a2[msk] - a1[msk])
        rel = absd / (np.abs(a1[msk]) + 1e-12)
        diffs.append((rel.mean(), name, absd.mean()))
    diffs.sort(reverse=True)
    print(f"\n----- {plat} {typ}: {len(shared)} shared vars -----")
    for rel, name, absd in diffs[:12]:
        print(f"   {name:22} mean|delta|={absd:.4g}  mean rel={rel * 100:.3f}%")
    if diffs:
        print(f"   overall mean relative change: {np.mean([d[0] for d in diffs]) * 100:.3f}%  "
              f"(0% = identical calibration between the two epochs)")


2 ADF files at ['20240430', '20240722']:
       0.0 MB  S2A DATAT 20240722  S2A_ADF_DATAT_20240722T000000_21000101T000000_20240722T000000.json
       0.0 MB  S2B DATAT 20240722  S2B_ADF_DATAT_20240722T000000_21000101T000000_20240722T000000.json

===== S2A DATAT 20240722  (0.0 MB) =====
  top-level keys: ['coords', 'attrs', 'dims', 'data_vars']
  attrs: {'legacy_documentation': {'attrib': {'name': 'DATA', 'type': 'gipp:A_DATATION_PARAMETERS'}, 'documentation': None, 'children': {'TRACE_MODE': {'attrib': {'name': 'TRACE_MODE', 'type': 'xs:boolean'}, 'documentation': 'NOT USED FOR GPP', 'children': {}}, 'THEORETICAL_LINE_PERIOD': {'attrib': {'name': 'THEORETICAL_LINE_PERIOD', 'type': 'misc:A_DOUBLE_WITH_MS_UNIT_ATTR'}, 'documentation': 'The theoretical line period for the acquisition of line of 10 m full-resolution image data', 'children': {}}, 'TIME_SHIFT_LIST': {'attrib': {'name': 'TIME_SHIFT_LIST'}, 'documentation': 'The time stamp is shifted taking into account a constant signed bias 

In [ ]:
import sys
import os
from pathlib import Path

data_store_dir = Path("/home/jovyan/data-store/synthetic-raw-generated")

os.chdir(data_store_dir)

!tree -d







.
├── input
│   ├── adf
│   ├── gipp
│   └── image
└── output

5 directories


: 

In [ ]:
# 27) data-store — yeni yapı (2026-07-07 düzenlemesi)
# esa-source/ (PPB + aux: gipp-xml, gipp-json, adf-eopf, psf) | synthetic-raw-generated/ (outputs/L0, calibration-campaign)
# msi-processor/ | archive-pre-ppb/ (eski her şey, non-destructive) | research/ (listing'ler)
!tree -d -L 3 ~/data-store